In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip -q install -U transformers datasets peft accelerate scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pan

In [4]:
%%writefile train_centralized_distilbert_lora_agnews.py
import os
import gc
import json
import time
import glob
import shutil
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel


@dataclass
class Config:
    # ===== Model / Data =====
    model_name: str = "distilbert-base-uncased"
    dataset_name: str = "ag_news"
    setting: str = "Centralized DistilBERT + LoRA"
    num_labels: int = 4
    seed: int = 42

    # ===== Output =====
    drive_root: str = "/content/drive/MyDrive/centralized_distilbert_lora_agnews"
    experiment_name: str = "centralized_distilbert_lora_agnews"

    # ===== Tokenization / Data =====
    max_length: int = 256
    validation_ratio: float = 0.1

    # ===== Training =====
    train_batch_size: int = 16
    eval_batch_size: int = 64
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.1

    # Không giới hạn cố định thực tế; dừng bằng early stopping
    max_epochs: int = 1000

    # ===== LoRA =====
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    target_modules: tuple = ("q_lin", "k_lin", "v_lin")

    # ===== Early stopping =====
    monitor: str = "accuracy"  # hoặc "macro_f1"
    greater_is_better: bool = True
    early_stopping_patience: int = 3
    min_delta: float = 0.0

    # ===== Stability for Colab =====
    fp16: bool = False
    num_workers: int = 0
    resume: bool = True
    save_latest_k: int = 2

    @property
    def exp_dir(self):
        return os.path.join(self.drive_root, self.experiment_name)

    @property
    def ckpt_dir(self):
        return os.path.join(self.exp_dir, "checkpoints")

    @property
    def best_model_dir(self):
        return os.path.join(self.exp_dir, "best_model")

    @property
    def final_model_dir(self):
        return os.path.join(self.exp_dir, "final_model")

    @property
    def history_json_path(self):
        return os.path.join(self.exp_dir, "history.json")

    @property
    def history_csv_path(self):
        return os.path.join(self.exp_dir, "history.csv")

    @property
    def config_path(self):
        return os.path.join(self.exp_dir, "config.json")

    @property
    def summary_path(self):
        return os.path.join(self.exp_dir, "run_summary.json")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: Config):
    os.makedirs(cfg.drive_root, exist_ok=True)
    os.makedirs(cfg.exp_dir, exist_ok=True)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    os.makedirs(cfg.best_model_dir, exist_ok=True)
    os.makedirs(cfg.final_model_dir, exist_ok=True)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_history_csv(path, history):
    if len(history) == 0:
        return
    pd.DataFrame(history).to_csv(path, index=False, encoding="utf-8")


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


def metric_improved(current, best, greater_is_better=True, min_delta=0.0):
    if greater_is_better:
        return current > (best + min_delta)
    return current < (best - min_delta)


def tokenize_function(examples, tokenizer, max_length):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
    )


def prepare_data(cfg: Config, tokenizer):
    ds = load_dataset(cfg.dataset_name)

    split_ds = ds["train"].train_test_split(
        test_size=cfg.validation_ratio,
        seed=cfg.seed,
        shuffle=True
    )

    train_ds = split_ds["train"]
    val_ds = split_ds["test"]

    tokenized_train = train_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )
    tokenized_val = val_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )

    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_val = tokenized_val.rename_column("label", "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_loader = DataLoader(
        tokenized_train,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        tokenized_val,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader


def create_lora_model(cfg: Config):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        target_modules=list(cfg.target_modules),
        modules_to_save=["classifier", "pre_classifier"],
    )

    model = get_peft_model(base_model, peft_config)
    return model


def list_checkpoints(ckpt_dir: str):
    ckpts = []
    if not os.path.exists(ckpt_dir):
        return ckpts

    for path in glob.glob(os.path.join(ckpt_dir, "checkpoint-epoch-*")):
        try:
            epoch_num = int(path.split("-")[-1])
            ckpts.append((epoch_num, path))
        except Exception:
            pass

    ckpts.sort(key=lambda x: x[0])
    return ckpts


def latest_checkpoint_path(ckpt_dir: str):
    ckpts = list_checkpoints(ckpt_dir)
    return ckpts[-1][1] if ckpts else None


def prune_old_checkpoints(ckpt_dir: str, keep_latest_k: int = 2):
    ckpts = list_checkpoints(ckpt_dir)
    while len(ckpts) > keep_latest_k:
        _, old_path = ckpts.pop(0)
        try:
            shutil.rmtree(old_path)
        except Exception:
            pass


def clear_dir_keep_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def save_training_artifacts(cfg: Config, save_dir: str, model, tokenizer):
    clear_dir_keep_dir(save_dir)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    save_json(os.path.join(save_dir, "model_info.json"), {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "num_labels": cfg.num_labels,
    })


def save_checkpoint(
    cfg,
    epoch,
    model,
    tokenizer,
    optimizer,
    scheduler,
    best_metric,
    patience_counter,
    history,
):
    ckpt_path = os.path.join(cfg.ckpt_dir, f"checkpoint-epoch-{epoch}")
    os.makedirs(ckpt_path, exist_ok=True)

    model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)

    state = {
        "epoch": epoch,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": best_metric,
        "patience_counter": patience_counter,
        "history": history,
        "config": asdict(cfg),
    }
    torch.save(state, os.path.join(ckpt_path, "training_state.pt"))

    with open(os.path.join(cfg.ckpt_dir, "latest_checkpoint.txt"), "w", encoding="utf-8") as f:
        f.write(ckpt_path)

    prune_old_checkpoints(cfg.ckpt_dir, cfg.save_latest_k)
    return ckpt_path


def load_latest_checkpoint(cfg, device="cpu"):
    ckpt_path = latest_checkpoint_path(cfg.ckpt_dir)
    if ckpt_path is None or not cfg.resume:
        return None, None, 0, None, 0, [], None, None

    state_path = os.path.join(ckpt_path, "training_state.pt")
    adapter_config_path = os.path.join(ckpt_path, "adapter_config.json")

    if not (os.path.exists(state_path) and os.path.exists(adapter_config_path)):
        print("Checkpoint files missing. Start from scratch.")
        return None, None, 0, None, 0, [], None, None

    print(f"Resuming from checkpoint: {ckpt_path}")

    base_model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels
    )
    model = PeftModel.from_pretrained(base_model, ckpt_path, is_trainable=True)
    model.to(device)

    state = torch.load(state_path, map_location=device)

    optimizer_state = state.get("optimizer_state_dict", None)
    scheduler_state = state.get("scheduler_state_dict", None)

    return (
        model,
        ckpt_path,
        state["epoch"],
        state.get("best_metric", None),
        state.get("patience_counter", 0),
        state.get("history", []),
        optimizer_state,
        scheduler_state,
    )


@torch.no_grad()
def evaluate(model, loader, device, use_amp=False, split_name="validation"):
    model.eval()

    total_loss = 0.0
    total_steps = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc=f"Evaluating {split_name}", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        if use_amp:
            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(**batch)
                loss = outputs.loss
                logits = outputs.logits
        else:
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item()
        total_steps += 1
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    return {
        "eval_loss": float(total_loss / max(total_steps, 1)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
        "macro_f1": float(f1_score(all_labels, all_preds, average="macro")),
    }


def verify_required_outputs(cfg: Config):
    required_paths = [
        cfg.history_json_path,
        cfg.history_csv_path,
        cfg.summary_path,
        cfg.config_path,
        os.path.join(cfg.best_model_dir, "adapter_config.json"),
        os.path.join(cfg.final_model_dir, "adapter_config.json"),
    ]

    missing = [p for p in required_paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError("Missing required output files:\n" + "\n".join(missing))


def train():
    cfg = Config()
    set_seed(cfg.seed)
    ensure_dirs(cfg)
    save_json(cfg.config_path, asdict(cfg))

    device = get_device()
    print("=" * 100)
    print("Device:", device)
    print("Experiment dir:", cfg.exp_dir)
    print("=" * 100)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    train_loader, val_loader = prepare_data(cfg, tokenizer)

    total_training_steps = len(train_loader) * cfg.max_epochs
    warmup_steps = int(total_training_steps * cfg.warmup_ratio)

    resumed = load_latest_checkpoint(cfg, device=device)

    if resumed[0] is not None:
        model, ckpt_path, start_epoch, best_metric, patience_counter, history, optimizer_state, scheduler_state = resumed
    else:
        model = create_lora_model(cfg).to(device)
        ckpt_path = None
        start_epoch = 0
        best_metric = None
        patience_counter = 0
        history = []
        optimizer_state = None
        scheduler_state = None

    total_params, trainable_params = count_parameters(model)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps
    )

    if optimizer_state is not None:
        optimizer.load_state_dict(optimizer_state)
    if scheduler_state is not None:
        scheduler.load_state_dict(scheduler_state)

    if best_metric is None:
        best_metric = -1e18 if cfg.greater_is_better else 1e18

    print(f"Start from epoch {start_epoch + 1}")
    print(f"Best {cfg.monitor} so far: {best_metric:.6f}")
    print(f"Patience counter: {patience_counter}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

    stopped_early = False
    use_amp = cfg.fp16 and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    epoch = start_epoch + 1
    while epoch <= cfg.max_epochs:
        model.train()
        epoch_start_time = time.time()
        running_loss = 0.0
        total_steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.cuda.amp.autocast(enabled=True):
                    outputs = model(**batch)
                    loss = outputs.loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                optimizer.step()

            scheduler.step()

            running_loss += loss.item()
            total_steps += 1

            if total_steps % 20 == 0:
                pbar.set_postfix(loss=f"{loss.item():.4f}")

        time_per_epoch = time.time() - epoch_start_time
        avg_train_loss = float(running_loss / max(total_steps, 1))

        val_metrics = evaluate(
            model,
            val_loader,
            device=device,
            use_amp=use_amp,
            split_name="validation"
        )

        current_metric = val_metrics[cfg.monitor]
        is_new_best = metric_improved(
            current=current_metric,
            best=best_metric,
            greater_is_better=cfg.greater_is_better,
            min_delta=cfg.min_delta
        )

        if is_new_best:
            best_metric = current_metric
            patience_counter = 0
            save_training_artifacts(
                cfg=cfg,
                save_dir=cfg.best_model_dir,
                model=model,
                tokenizer=tokenizer
            )
        else:
            patience_counter += 1

        epoch_result = {
            "epoch": int(epoch),
            "train_loss": float(avg_train_loss),
            "eval_loss": float(val_metrics["eval_loss"]),
            "accuracy": float(val_metrics["accuracy"]),
            "macro_f1": float(val_metrics["macro_f1"]),
            "time_per_epoch": float(time_per_epoch),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": cfg.model_name,
            "dataset_name": cfg.dataset_name,
            "setting": cfg.setting,
            "best_metric_so_far": float(best_metric),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }

        history.append(epoch_result)

        save_json(cfg.history_json_path, history)
        save_history_csv(cfg.history_csv_path, history)

        ckpt_path = save_checkpoint(
            cfg=cfg,
            epoch=epoch,
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            best_metric=best_metric,
            patience_counter=patience_counter,
            history=history,
        )

        print("-" * 100)
        print("EPOCH LOG:")
        print(json.dumps(epoch_result, indent=2))
        print("Checkpoint saved:", ckpt_path)
        print("-" * 100)

        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping triggered: no improvement in {cfg.early_stopping_patience} consecutive epochs.")
            stopped_early = True
            break

        epoch += 1

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    save_training_artifacts(
        cfg=cfg,
        save_dir=cfg.final_model_dir,
        model=model,
        tokenizer=tokenizer
    )

    run_summary = {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "monitor_metric": cfg.monitor,
        "best_metric_so_far": float(best_metric),
        "num_epochs_completed": int(len(history)),
        "stopped_early": bool(stopped_early),
        "best_model_dir": cfg.best_model_dir,
        "final_model_dir": cfg.final_model_dir,
        "latest_checkpoint": latest_checkpoint_path(cfg.ckpt_dir),
        "history_json": cfg.history_json_path,
        "history_csv": cfg.history_csv_path,
    }
    save_json(cfg.summary_path, run_summary)

    verify_required_outputs(cfg)

    print("=" * 100)
    print("OUTPUT DIRECTORY          :", cfg.exp_dir)
    print("Best model dir            :", cfg.best_model_dir)
    print("Final model dir           :", cfg.final_model_dir)
    print("Checkpoints dir           :", cfg.ckpt_dir)
    print("History JSON              :", cfg.history_json_path)
    print("History CSV               :", cfg.history_csv_path)
    print("Run summary               :", cfg.summary_path)
    print("=" * 100)
    print("Done.")


if __name__ == "__main__":
    train()

Overwriting train_centralized_distilbert_lora_agnews.py


In [5]:
!python train_centralized_distilbert_lora_agnews.py

Device: cuda
Experiment dir: /content/drive/MyDrive/centralized_distilbert_lora_agnews/centralized_distilbert_lora_agnews
Map: 100% 12000/12000 [00:03<00:00, 3022.03 examples/s]
Resuming from checkpoint: /content/drive/MyDrive/centralized_distilbert_lora_agnews/centralized_distilbert_lora_agnews/checkpoints/checkpoint-epoch-10
Loading weights: 100% 100/100 [00:00<00:00, 1673.42it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:c